In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from pathlib import Path
import numpy as np
import sys
import importlib
import matplotlib.pyplot as plt

In [2]:
PROJECT_ROOT = Path().resolve().parent

sys.path.append(str(PROJECT_ROOT / "src"))

import model_utils
importlib.reload(model_utils)
from model_utils import run_nested_cv

In [3]:
sample_fp = PROJECT_ROOT / "outputs" / "sample_points_indices.csv"
samples = pd.read_csv(sample_fp)

samples = samples.dropna(subset="lc")

pred_cols = ['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B8', 'NDVI', 'NDWI', 'MNDWI', 'NDCI', 'NDMI', 'NDBI', 'NDAVI', 'AWEIp95']
metadata_cols = ["label_id","location","class_int", "class_label","obs_date"]


y = samples['lc'].values # values gives you numpy array
X = samples[pred_cols].values
metadata = samples[metadata_cols]
groups = samples['location'].values 


In [4]:
AWEI = "AWEIp95" in pred_cols

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

param_grid = {
    'model__n_estimators': [100,200,500], # have to prefix the keys with the pipeline stepname followed by __
    'model__max_depth': [3,5,10,20,None],
    'model__min_samples_leaf': [1, 5, 10]
    }

results_df, predictions_df, inner_df = run_nested_cv(X, y, groups, metadata, pipe, param_grid, n_jobs= 4)

results_df["AWEIp95"] = AWEI
inner_df["AWEIp95"] = AWEI
predictions_df["AWEIp95"] = AWEI

model = "RF" + f"_AWEI_{AWEI}"


predictions_fp = PROJECT_ROOT / "outputs" /"RF_outputs"/ f"{model}_predictions_.csv"
predictions_df.to_csv(predictions_fp)
results_fp = PROJECT_ROOT / "outputs"/ "RF_outputs"/ f"{model}_outer_results.csv"
results_df.to_csv(results_fp)
inner_fp = PROJECT_ROOT / "outputs"/"RF_outputs"/ f"{model}_inner_results.csv"
inner_df.to_csv(inner_fp)

F1 Binary:  0.937 +/- 0.036
F1 Macro:   0.937 +/- 0.033
Accuracy:   0.937 +/- 0.033


In [5]:
sample_fp = PROJECT_ROOT / "outputs" / "sample_points_indices.csv"
samples = pd.read_csv(sample_fp)

samples = samples.dropna(subset="lc")

pred_cols = ['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B8', 'NDVI', 'NDWI', 'MNDWI', 'NDCI', 'NDMI', 'NDBI', 'NDAVI']
metadata_cols = ["label_id","location","class_int", "class_label","obs_date"]


y = samples['lc'].values # values gives you numpy array
X = samples[pred_cols].values
metadata = samples[metadata_cols]
groups = samples['location'].values 


In [6]:
AWEI = "AWEIp95" in pred_cols

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

param_grid = {
    'model__n_estimators': [100,200,500], # have to prefix the keys with the pipeline stepname followed by __
    'model__max_depth': [3,5,10,20,None],
    'model__min_samples_leaf': [1, 5, 10]
    }

results_df, predictions_df, inner_df = run_nested_cv(X, y, groups, metadata, pipe, param_grid, n_jobs= 4)

results_df["AWEIp95"] = AWEI
inner_df["AWEIp95"] = AWEI
predictions_df["AWEIp95"] = AWEI

model = "RF" + f"_AWEI_{AWEI}"


predictions_fp = PROJECT_ROOT / "outputs" /"RF_outputs"/ f"{model}_predictions_.csv"
predictions_df.to_csv(predictions_fp)
results_fp = PROJECT_ROOT / "outputs"/ "RF_outputs"/ f"{model}_outer_results.csv"
results_df.to_csv(results_fp)
inner_fp = PROJECT_ROOT / "outputs"/"RF_outputs"/ f"{model}_inner_results.csv"
inner_df.to_csv(inner_fp)

F1 Binary:  0.906 +/- 0.043
F1 Macro:   0.904 +/- 0.042
Accuracy:   0.905 +/- 0.041


In [12]:
results_df

,test_location,test_f1,test_f1_macro,test_accuracy,test_precision,test_recall,test_auroc,best_n_estimators,best_max_depth,best_min_samples_leaf,train_f1,train_f1_macro,train_accuracy,f1_gap,f1_macro_gap,accuracy_gap,AWEIp95
0,Hartbeespoort,0.949744,0.951131,0.951171,0.976793,0.924152,0.951131,200,5,5,0.932974,0.931793,0.931813,-0.016770,-0.019339,-0.019358,False
1,Inle,0.917116,0.914525,0.914603,0.899038,0.935936,0.914397,100,5,1,0.939117,0.937603,0.937640,0.022001,0.023078,0.023036,False
2,Mula,0.820879,0.840302,0.842664,0.914321,0.744766,0.839642,200,10,5,0.966457,0.965919,0.965927,0.145578,0.125617,0.123263,False
3,RawaPening,0.940358,0.940058,0.940060,0.936634,0.944112,0.940056,100,5,5,0.934441,0.933409,0.933425,-0.005916,-0.006650,-0.006635,False
4,Rodman,0.899734,0.877892,0.881799,0.822384,0.993144,0.873676,100,5,5,0.940859,0.940225,0.940232,0.041126,0.062333,0.058433,False
5,Valsequillo,0.940495,0.940306,0.940306,0.945975,0.935079,0.940354,200,5,5,0.935133,0.934291,0.934302,-0.005362,-0.006015,-0.006005,False
6,Vembanad,0.926390,0.926573,0.926573,0.929648,0.923154,0.926577,200,5,5,0.935668,0.933957,0.934002,0.009278,0.007384,0.007428,False
7,Winam,0.854196,0.843720,0.844422,0.804060,0.911000,0.844389,100,5,10,0.939936,0.939199,0.939208,0.085740,0.095479,0.094786,False


In [8]:
# predictions_fp = PROJECT_ROOT / "outputs" / "nested_cv_predictions.csv"

# predictions_df = pd.read_csv(predictions_fp, index_col=0) 
# predictions_df.head()

In [9]:
# conf_matrix = confusion_matrix(predictions_df["y_true"], predictions_df["y_pred"])

# conf_m_disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=["negative","positive"] )
# conf_m_disp.plot()
# plt.show()

In [10]:
# fig, axes = plt.subplots(2, 4, figsize=(16, 8))  # adjust grid to number of locations
# axes = axes.flatten()

# for ax, (location, group) in zip(axes, predictions_df.groupby('location')):
#     cm = confusion_matrix(group['y_true'], group['y_pred'])
#     disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['0', '1'])
#     disp.plot(ax=ax)
#     ax.set_title(location)

# plt.tight_layout()
# plt.show()

In [11]:
# for location, group in predictions_df.groupby('location'):
#     print(f"\n{location}")
#     print(pd.crosstab(group['class_int'], group['y_pred'], normalize='index'))